# 02 — Steam Market Analysis

## Objective

This notebook analyzes the main characteristics of the Steam videogame market using the prepared game-level dataset created in the data-preparation stage.

The objective is to identify market patterns that can help Ubisoft better understand the competitive environment before releasing a new videogame on Steam.

The analysis focuses on the following business questions:

1. How has the number of videogame releases evolved over time?
2. Which publishers have the largest presence on Steam?
3. How is the market structured in terms of paid and free-to-play games?
4. What are the main pricing and discount patterns?
5. How widely do games support multiple languages?
6. How common are age restrictions?
7. Which games and market segments show the strongest player reception and engagement?

Because direct revenue and exact sales information are unavailable, review volume, concurrent users, and ownership estimates are used cautiously as **popularity or commercial-reach proxies** rather than direct measures of financial success.

---

## Methodology

The analysis follows a **Spark-first exploratory approach**.

Aggregations, filtering, grouping, and feature comparisons are performed using PySpark DataFrame operations. Databricks visualizations are created from aggregated results rather than by converting the complete dataset to pandas.

Particular attention is given to highly skewed variables such as prices, review counts, concurrent users, and ownership estimates. Extreme observations are not automatically treated as errors; medians, thresholds, or appropriate visualization scales are used when necessary to avoid misleading interpretations.

The notebook focuses on market-level patterns. Detailed genre and platform analysis is intentionally reserved for the following notebook.

## 1. Environment and Prepared Data Loading

The validated game-level dataset produced in the previous notebook is loaded from the persisted Delta table.

Using the prepared dataset ensures that the market analysis relies on the same documented cleaning and feature-engineering decisions without repeating the complete preparation pipeline.

In [0]:
# PySpark imports
# ---------------------------------------------------------------------------

from pyspark.sql import functions as F


# Load the prepared game-level dataset
# ---------------------------------------------------------------------------

TABLE_NAME = "steam_games_prepared"

steam_games_df = spark.table(TABLE_NAME)

print(f"Steam videogames: {steam_games_df.count():,}")
print(f"Analytical columns: {len(steam_games_df.columns)}")

Steam videogames: 55,690
Analytical columns: 37


In [0]:
# Verify the analytical dataset loaded correctly
# ---------------------------------------------------------------------------

display(
    steam_games_df.select(
        "appid",
        "name",
        "publisher",
        "release_year",
        "initial_price",
        "discount_pct",
        "total_reviews",
        "positive_review_ratio",
        "owner_midpoint",
        "language_count",
        "required_age",
        "platform_count",
    ).limit(5)
)

appid,name,publisher,release_year,initial_price,discount_pct,total_reviews,positive_review_ratio,owner_midpoint,language_count,required_age,platform_count
10,Counter-Strike,Valve,2000,9.99,0.0,206414,0.9748127549487923,1.5E7,8,0,3
1000000,ASCENXION,PsychoFlux Entertainment,2021,9.99,0.0,32,0.84375,10000.0,3,0,1
1000010,Crown Trick,"Team17, NEXT Studios",2020,19.99,70.0,4678,0.8619067977768277,350000.0,9,0,1
1000030,"Cook, Serve, Delicious! 3?!",Vertigo Gaming Inc.,2020,19.99,0.0,1690,0.9319526627218935,150000.0,1,0,2
1000040,细胞战争,DoubleC Games,2019,1.99,0.0,1,0.0,10000.0,1,0,1


## 2. Market Evolution Over Time

The first analysis examines how the number of videogames released on Steam has evolved over time.

Understanding release activity provides a high-level view of the development of the Steam marketplace and the competitive environment faced by publishers. A growing number of annual releases may indicate increasing platform activity and competition for player attention.

Only games with a valid parsed release year are included in this time-based analysis. Games without a complete release date remain in the prepared dataset but cannot contribute to annual release counts.

### Annual Number of Videogame Releases

The number of unique videogames released each year is calculated using Spark aggregations.

`countDistinct(appid)` is used rather than a simple row count so that the metric explicitly represents the number of unique Steam applications released in each year.

In [0]:
# Aggregate the number of unique videogame releases by year
# ---------------------------------------------------------------------------

releases_by_year_df = (
    steam_games_df
    .filter(F.col("release_year").isNotNull())
    .groupBy("release_year")
    .agg(
        F.countDistinct("appid").alias("games_released")
    )
    .orderBy("release_year")
)

display(releases_by_year_df)

release_year,games_released
1997,2
1998,1
1999,3
2000,2
2001,4
2002,1
2003,3
2004,6
2005,6
2006,61


Databricks visualization. Run in Databricks to view.

### 2. Key Findings — Release Activity

The Steam videogame dataset shows a substantial expansion in annual release activity over time.

Release volumes remained relatively limited during the earlier years represented in the dataset, before accelerating strongly from the mid-2010s. Annual releases increased from **1,550 games in 2014** to **2,565 in 2015**, then continued rising to **6,006 in 2017** and **7,663 in 2018**.

After a temporary decrease to **6,949 releases in 2019**, release activity increased again, reaching **8,287 games in 2020** and a dataset peak of **8,805 games in 2021**.

The dataset records **7,451 releases in 2022**. However, this should not be interpreted as evidence of a market decline because the latest valid release date in the dataset is **2022-11-11**. The 2022 observation therefore represents an incomplete calendar year.

Overall, the data indicates that Steam developed into an increasingly crowded publishing environment, particularly from the mid-2010s onward. For Ubisoft, this suggests that releasing a game on Steam involves competing for player attention within a marketplace characterized by a high volume of new titles.

These results describe the release activity represented in the available dataset and should not be interpreted as a complete historical record of every Steam release.

## 3. Publisher Landscape

The Steam marketplace includes games released by a wide range of publishers, from large established companies to independent studios and self-published developers.

This section examines publisher presence based on the number of unique videogames associated with each publisher in the dataset.

Publisher presence should not be interpreted as market share in terms of revenue or players. A publisher with many released games does not necessarily generate more sales or engagement than a publisher with a smaller catalogue.

### Most Active Publishers

The most represented publishers are identified by counting the number of unique Steam applications associated with each publisher.

Games without publisher information are excluded from this ranking because they cannot be reliably assigned to a publisher.


In [0]:
# Identify publishers with the largest Steam catalogues
# ---------------------------------------------------------------------------

top_publishers_df = (
    steam_games_df
    .filter(F.col("publisher").isNotNull())
    .groupBy("publisher")
    .agg(
        F.countDistinct("appid").alias("games_published")
    )
    .orderBy(F.desc("games_published"))
    .limit(15)
)

display(top_publishers_df)

publisher,games_published
Big Fish Games,423
8floor,202
SEGA,165
Strategy First,151
Square Enix,141
Choice of Games,140
Sekai Project,132
HH-Games,132
Ubisoft,127
Laush Studio,126


Databricks visualization. Run in Databricks to view.

In [0]:
# Measure the diversity of publishers represented in the dataset
# ---------------------------------------------------------------------------

publisher_summary_df = (
    steam_games_df
    .filter(F.col("publisher").isNotNull())
    .agg(
        F.countDistinct("publisher").alias("distinct_publishers"),
        F.countDistinct("appid").alias("games_with_publisher")
    )
)

display(publisher_summary_df)

distinct_publishers,games_with_publisher
29833,55556


### 3. Key Findings — Publisher Landscape

The Steam publisher landscape is highly fragmented. Among the **55,556 games with available publisher information**, the dataset contains **29,833 distinct publisher entries**, indicating a large and diverse publishing ecosystem.

**Big Fish Games** has the largest catalogue represented in the dataset with **423 games**, followed by **8floor (202)** and **SEGA (165)**. Other highly represented publishers include Strategy First, Square Enix, Choice of Games, and Sekai Project.

**Ubisoft ranks ninth among the most represented publisher entries, with 127 games** in the dataset. This confirms that Ubisoft already has a substantial catalogue presence on Steam, although catalogue size alone should not be interpreted as commercial performance.

Even the most represented publisher accounts for fewer than 1% of games with known publisher information. The distribution therefore suggests that Steam is not dominated by a small number of publishers in terms of catalogue volume. Instead, it contains a very large number of publishers with comparatively small catalogues.

For Ubisoft, this reinforces the competitive picture observed in the release analysis: competition for visibility comes not only from other large publishers but also from a broad and fragmented ecosystem of smaller publishers and independent releases.

Publisher values are analyzed as represented in the source publisher field. Because publisher information may contain combined or differently formatted publisher names, these results describe catalogue presence rather than a definitive measure of corporate market share.

## 4. Pricing and Business Model

Pricing is an important component of videogame positioning on Steam. This section examines the prevalence of free versus paid games and the price distribution of paid titles.

Free and paid games are analyzed separately because including zero-priced games in general price statistics can distort the interpretation of the commercial pricing structure.

The analysis uses initial price to identify games originally offered for free, while current price and discount information are considered separately when examining promotional activity.

### 4.1 Free vs Paid Games

The market is first divided between games with an initial price of zero and paid games.

Both counts and percentages are calculated to provide an interpretable view of the relative prevalence of each business model in the dataset.

In [0]:
# Compare free and paid videogames
# ---------------------------------------------------------------------------

free_paid_df = (
    steam_games_df
    .withColumn(
        "pricing_model",
        F.when(F.col("is_free"), "Free")
        .otherwise("Paid")
    )
    .groupBy("pricing_model")
    .agg(
        F.countDistinct("appid").alias("games")
    )
    .withColumn(
        "percentage",
        F.round(
            F.col("games") / F.lit(steam_games_df.count()) * 100,
            2
        )
    )
    .orderBy(F.desc("games"))
)

display(free_paid_df)

pricing_model,games,percentage
Paid,47911,86.03
Free,7779,13.97


Databricks visualization. Run in Databricks to view.

### 4.2 Price Distribution of Paid Games

Price statistics.

In [0]:
# Summarize initial prices among paid games
# ---------------------------------------------------------------------------

paid_price_summary_df = (
    steam_games_df
    .filter(F.col("initial_price") > 0)
    .agg(
        F.countDistinct("appid").alias("paid_games"),
        F.round(F.avg("initial_price"), 2).alias("average_price"),
        F.round(
            F.expr("percentile_approx(initial_price, 0.5)"),
            2
        ).alias("median_price"),
        F.round(
            F.expr("percentile_approx(initial_price, 0.25)"),
            2
        ).alias("q1_price"),
        F.round(
            F.expr("percentile_approx(initial_price, 0.75)"),
            2
        ).alias("q3_price"),
        F.max("initial_price").alias("max_price"),
    )
)

display(paid_price_summary_df)

paid_games,average_price,median_price,q1_price,q3_price,max_price
47911,9.27,5.99,2.99,11.99,999.0


### 4.1–4.2 Key Findings — Pricing Structure

The Steam videogame catalogue is predominantly composed of paid titles. Among the **55,690 games** in the analytical dataset, **47,911 (86.03%)** have a positive initial price, while **7,779 (13.97%)** are classified as free based on an initial price of zero.

This indicates that free-to-play titles represent a meaningful but minority share of the catalogue. For Ubisoft, the results suggest that paid releases remain the dominant pricing model among the games represented in the dataset.

Among paid games, the **median initial price is 5.99**, compared with an **average of 9.27**. The middle 50% of paid titles are priced between **2.99 and 11.99**.

The average being higher than the median indicates a right-skewed price distribution, consistent with the presence of a smaller number of comparatively expensive titles. The maximum observed initial price is **999**, which was previously validated as a legitimate extreme observation rather than a conversion error.

Consequently, the median and quartiles provide a more representative description of typical catalogue pricing than the average alone. These figures describe the overall Steam catalogue and should not be interpreted as an optimal price recommendation for a future Ubisoft release, since appropriate pricing may also depend on genre, positioning, production scale, and other product characteristics.

Consequently, the median and quartiles provide a more representative description of typical catalogue pricing than the average alone. These figures describe the overall Steam catalogue and should not be interpreted as an optimal price recommendation for a future Ubisoft release, since appropriate pricing may also depend on genre, positioning, production scale, and other product characteristics.

### 4.3 Discount Activity

Steam is also characterized by promotional pricing. This analysis measures how many games are currently discounted in the dataset and examines the magnitude of observed discounts.

The discount variable represents the promotional state captured in the dataset rather than the complete historical discount strategy of each game. The results therefore provide a snapshot of discount activity rather than a measure of how frequently games are discounted over their lifetime.

In [0]:
# Summarize current discount activity
# ---------------------------------------------------------------------------

discount_summary_df = (
    steam_games_df
    .agg(
        F.countDistinct("appid").alias("total_games"),
        F.sum(
            F.when(F.col("is_discounted"), 1).otherwise(0)
        ).alias("discounted_games"),
        F.round(
            F.avg(
                F.when(
                    F.col("is_discounted"),
                    F.col("discount_pct")
                )
            ),
            2
        ).alias("average_discount_pct"),
        F.expr(
            """
            percentile_approx(
                CASE WHEN is_discounted THEN discount_pct END,
                0.5
            )
            """
        ).alias("median_discount_pct"),
        F.max("discount_pct").alias("max_discount_pct")
    )
    .withColumn(
        "discounted_percentage",
        F.round(
            F.col("discounted_games") / F.col("total_games") * 100,
            2
        )
    )
)

display(discount_summary_df)

total_games,discounted_games,average_discount_pct,median_discount_pct,max_discount_pct,discounted_percentage
55690,2518,57.59,60.0,90.0,4.52


In [0]:
# Compare discount activity by original pricing model
# ---------------------------------------------------------------------------

discount_by_model_df = (
    steam_games_df
    .withColumn(
        "pricing_model",
        F.when(F.col("is_free"), "Free")
        .otherwise("Paid")
    )
    .groupBy("pricing_model")
    .agg(
        F.countDistinct("appid").alias("games"),
        F.sum(
            F.when(F.col("is_discounted"), 1).otherwise(0)
        ).alias("discounted_games")
    )
    .withColumn(
        "discounted_percentage",
        F.round(
            F.col("discounted_games") / F.col("games") * 100,
            2
        )
    )
    .orderBy(F.desc("games"))
)

display(discount_by_model_df)


pricing_model,games,discounted_games,discounted_percentage
Paid,47911,2518,5.26
Free,7779,0,0.0


### 4.3 Key Findings — Discount Activity

Current discounts are relatively uncommon across the complete Steam catalogue represented in the dataset. Only **2,518 of 55,690 games (4.52%)** are discounted at the time captured by the data.

Discount activity is entirely concentrated among originally paid games. Among the **47,911 paid titles**, **2,518 (5.26%)** are currently discounted, while none of the 7,779 games classified as initially free have a recorded discount.

Although only a small proportion of the catalogue is discounted at this particular point in time, the discounts applied to those games are substantial. Discounted titles have an **average discount of 57.59%** and a **median discount of 60%**, with the largest observed discount reaching **90%**.

This suggests that promotional pricing, when active, can involve substantial price reductions. However, the dataset represents only a snapshot of current discount status and does not contain historical promotion frequency or duration. The **4.52% figure should therefore not be interpreted as the proportion of Steam games that ever use discounts**.

For Ubisoft, the results indicate that discounting is a potentially significant promotional mechanism, but this dataset alone cannot determine the optimal discount level, timing, or frequency for a future release.

## 5. Language Availability

Steam serves an international player base, making language availability an important dimension of product accessibility.

This section examines how many languages games support and identifies the languages most frequently listed across the videogame catalogue.

The analysis distinguishes between:

- **language breadth**, measured by the number of supported languages per game;
- **individual language representation**, measured after transforming the language arrays into one row per game-language combination.

Games without available language information are excluded only from the corresponding language analysis.

In [0]:
# Summarize the number of supported languages per game
# ---------------------------------------------------------------------------

language_summary_df = (
    steam_games_df
    .filter(F.col("language_count").isNotNull())
    .agg(
        F.countDistinct("appid").alias("games_with_language_data"),
        F.round(F.avg("language_count"), 2).alias("average_languages"),
        F.expr(
            "percentile_approx(language_count, 0.5)"
        ).alias("median_languages"),
        F.expr(
            "percentile_approx(language_count, 0.25)"
        ).alias("q1_languages"),
        F.expr(
            "percentile_approx(language_count, 0.75)"
        ).alias("q3_languages"),
        F.max("language_count").alias("max_languages")
    )
)

display(language_summary_df)

games_with_language_data,average_languages,median_languages,q1_languages,q3_languages,max_languages
55680,3.63,1,1,4,40


In [0]:
# Create one row per game-language combination
# ---------------------------------------------------------------------------

game_languages_df = (
    steam_games_df
    .filter(F.col("languages").isNotNull())
    .select(
        "appid",
        F.explode("languages").alias("language")
    )
    .filter(
        F.col("language").isNotNull()
        & (F.trim(F.col("language")) != "")
    )
)

In [0]:
# Identify the most frequently supported languages
# ---------------------------------------------------------------------------

top_languages_df = (
    game_languages_df
    .groupBy("language")
    .agg(
        F.countDistinct("appid").alias("games")
    )
    .orderBy(F.desc("games"))
    .limit(15)
)

display(top_languages_df)

language,games
English,55116
German,14019
French,13426
Russian,12922
Simplified Chinese,12782
Spanish - Spain,12233
Japanese,10368
Italian,9304
Portuguese - Brazil,6750
Korean,6600


Databricks visualization. Run in Databricks to view.

### 5.1–5.2 Key Findings — Language Availability

Language availability varies substantially across the Steam videogame catalogue.

Among the **55,680 games with available language information**, the average game supports **3.63 languages**, while the median is only **1 language**. The first quartile is also 1 language and the third quartile is 4 languages, indicating that at least half of the games in the dataset support only a single language, while a smaller group of highly localized titles raises the overall average. The most extensively localized game supports **40 languages**.

After transforming the language arrays into individual game-language observations, **English is by far the most frequently supported language**, appearing in **55,116 games** with language information. Other frequently represented languages include German (14,019 games), French (13,426), Russian (12,922), Simplified Chinese (12,782), and Spanish - Spain (12,233).

Asian-language support is also visible among the most common languages, particularly Japanese, Simplified Chinese, Korean, and Traditional Chinese. Portuguese appears separately in Brazilian and Portuguese variants, reflecting the source dataset's regional language classifications.

For Ubisoft, these results highlight a distinction between basic accessibility and broad localization. English support is nearly universal in the dataset, whereas extensive multilingual support is considerably less common. Language breadth may therefore represent an important product-positioning dimension for games targeting an international audience.

The analysis measures the presence of listed language support only. It does not distinguish between interface, subtitle, and full-audio localization quality.

## 6. Age Restrictions

Age requirements provide an additional perspective on the accessibility and positioning of games available on Steam.

The source field is primarily numerical but contains a very small number of non-numeric rating labels. The prepared numerical `required_age` variable therefore contains null values for records that cannot be represented consistently as a minimum age.

This analysis examines the distribution of the numerical age requirements while preserving those source limitations.

In [0]:
# Inspect the distribution of numerical age requirements
# ---------------------------------------------------------------------------

age_distribution_df = (
    steam_games_df
    .groupBy("required_age")
    .agg(
        F.countDistinct("appid").alias("games")
    )
    .orderBy("required_age")
)

display(age_distribution_df)

required_age,games
null,3
0,55029
3,3
5,1
6,4
7,2
8,3
9,1
10,7
12,32


In [0]:
# Compare games with and without a numerical age restriction
# ---------------------------------------------------------------------------

age_restriction_summary_df = (
    steam_games_df
    .withColumn(
        "age_requirement",
        F.when(
            F.col("required_age").isNull(),
            "Unknown / non-numeric"
        )
        .when(
            F.col("required_age") == 0,
            "No numerical restriction"
        )
        .otherwise("Numerical age restriction")
    )
    .groupBy("age_requirement")
    .agg(
        F.countDistinct("appid").alias("games")
    )
    .withColumn(
        "percentage",
        F.round(
            F.col("games") / F.lit(steam_games_df.count()) * 100,
            2
        )
    )
    .orderBy(F.desc("games"))
)

display(age_restriction_summary_df)

age_requirement,games,percentage
No numerical restriction,55029,98.81
Numerical age restriction,658,1.18
Unknown / non-numeric,3,0.01


### 6. Key Findings — Age Restrictions

Numerical age requirements are uncommon in the Steam catalogue represented by the dataset.

Among the **55,690 videogames**, **55,029 (98.81%)** have a recorded numerical age requirement of zero. Only **658 games (1.18%)** contain a positive numerical value, while **3 games (0.01%)** contain non-numeric source values that could not be converted to the numerical feature.

Among games with positive numerical requirements, values such as 15, 16, 17, and 18 appear in the data. However, the field also contains unusual values such as **35 and 180**, indicating that the variable should not be assumed to represent a fully standardized age-rating system in every record.

For this reason, the analysis uses the field primarily to distinguish games with no recorded numerical restriction from games containing a positive numerical requirement. Individual values are not interpreted as definitive regulatory age classifications without additional rating-system information.

Overall, the available data suggests that explicit numerical age requirements are recorded for only a small minority of games. This variable therefore provides limited evidence for broader market-positioning decisions and is not analyzed further.

## 7. Player Reception and Popularity

Player reviews and engagement indicators provide useful signals about how games are received and how much attention they attract on Steam.

The dataset does not contain direct revenue or exact sales information. Therefore, the analysis distinguishes between:

- **player reception**, represented by the proportion of recorded reviews that are positive;
- **popularity and engagement proxies**, represented by review volume, concurrent users, and approximate ownership information.

These indicators should not be interpreted as direct measures of profitability or commercial success.

A minimum review-volume requirement is also necessary when comparing positive-review ratios. Games with only a few reviews can otherwise appear among the highest-rated titles despite having very limited evidence of broad player reception.

### 7.1 Review Volume

In [0]:
# Summarize the distribution of total review volume
# ---------------------------------------------------------------------------

review_distribution_df = (
    steam_games_df
    .agg(
        F.countDistinct("appid").alias("total_games"),
        F.sum(
            F.when(F.col("total_reviews") == 0, 1).otherwise(0)
        ).alias("games_without_reviews"),
        F.round(F.avg("total_reviews"), 2).alias("average_reviews"),
        F.expr(
            "percentile_approx(total_reviews, 0.5)"
        ).alias("median_reviews"),
        F.expr(
            "percentile_approx(total_reviews, 0.75)"
        ).alias("q3_reviews"),
        F.expr(
            "percentile_approx(total_reviews, 0.90)"
        ).alias("p90_reviews"),
        F.expr(
            "percentile_approx(total_reviews, 0.95)"
        ).alias("p95_reviews"),
        F.expr(
            "percentile_approx(total_reviews, 0.99)"
        ).alias("p99_reviews"),
        F.max("total_reviews").alias("max_reviews")
    )
)

display(review_distribution_df)

total_games,games_without_reviews,average_reviews,median_reviews,q3_reviews,p90_reviews,p95_reviews,p99_reviews,max_reviews
55690,163,1712.61,26,144,918,2940,25904,6730438


In [0]:
# Evaluate candidate review-volume thresholds
# ---------------------------------------------------------------------------

review_thresholds_df = (
    steam_games_df
    .agg(
        F.sum(
            F.when(F.col("total_reviews") >= 100, 1).otherwise(0)
        ).alias("games_100_plus_reviews"),
        F.sum(
            F.when(F.col("total_reviews") >= 500, 1).otherwise(0)
        ).alias("games_500_plus_reviews"),
        F.sum(
            F.when(F.col("total_reviews") >= 1000, 1).otherwise(0)
        ).alias("games_1000_plus_reviews"),
        F.sum(
            F.when(F.col("total_reviews") >= 5000, 1).otherwise(0)
        ).alias("games_5000_plus_reviews")
    )
)

display(review_thresholds_df)

games_100_plus_reviews,games_500_plus_reviews,games_1000_plus_reviews,games_5000_plus_reviews
16332,7767,5311,1972


In [0]:
# Validate the proposed minimum review threshold
# ---------------------------------------------------------------------------

review_750_threshold_df = (
    steam_games_df
    .agg(
        F.sum(
            F.when(F.col("total_reviews") >= 750, 1).otherwise(0)
        ).alias("games_750_plus_reviews")
    )
    .withColumn(
        "percentage_of_games",
        F.round(
            F.col("games_750_plus_reviews")
            / F.lit(steam_games_df.count()) * 100,
            2
        )
    )
)

display(review_750_threshold_df)

games_750_plus_reviews,percentage_of_games
6216,11.16


### 7.1 Key Findings — Review Volume

Review activity is highly uneven across the Steam videogame catalogue.

The median game has only **26 recorded reviews**, while the average reaches **1,712.61 reviews**. This substantial difference indicates a strongly right-skewed distribution in which a relatively small number of highly visible games accumulate very large review volumes.

The upper percentiles reinforce this pattern. The 75th percentile contains 144 reviews, the 90th percentile 918 reviews, and the 95th percentile 2,940 reviews. At the 99th percentile, review volume reaches 25,904, while the maximum observed value exceeds **6.7 million reviews**.

Only **163 games** have no recorded reviews, but having reviews does not necessarily imply substantial player exposure. For example, 16,332 games have at least 100 reviews, while progressively fewer games reach the higher review-volume thresholds.

For comparisons based on positive-review ratio, a minimum threshold of **750 total reviews** is used. This threshold is substantially above the typical review volume in the dataset and approaches the 90th percentile of 918 reviews. It retains **6,216 games (11.16% of the catalogue)** while reducing the influence of titles whose apparently high reception is based on limited review evidence.

The 750-review cutoff is an analytical filtering choice rather than a universal definition of a well-established game. Its purpose is to balance sufficient review evidence with the retention of a meaningful comparison group.

Review volume is interpreted as a proxy for player attention and popularity rather than as a direct measure of sales or profitability.

### 7.2 Most Reviewed Games

In [0]:
# Identify games with the largest recorded review volumes
# ---------------------------------------------------------------------------

most_reviewed_games_df = (
    steam_games_df
    .select(
        "appid",
        "name",
        "publisher",
        "total_reviews",
        "positive_review_ratio"
    )
    .withColumn(
        "positive_review_pct",
        F.round(F.col("positive_review_ratio") * 100, 2)
    )
    .orderBy(F.desc("total_reviews"))
    .limit(15)
)

display(most_reviewed_games_df)

appid,name,publisher,total_reviews,positive_review_ratio,positive_review_pct
730,Counter-Strike: Global Offensive,Valve,6730438,0.8830547135268165,88.31
578080,PUBG: BATTLEGROUNDS,"KRAFTON, Inc.",2093876,0.5661084992616564,56.61
570,Dota 2,Valve,1852811,0.828414231133127,82.84
271590,Grand Theft Auto V,Rockstar Games,1442644,0.852091714934523,85.21
359550,Tom Clancy's Rainbow Six Siege,Ubisoft,1086157,0.8681157512219688,86.81
105600,Terraria,Re-Logic,1037091,0.978420408623737,97.84
440,Team Fortress 2,Valve,903830,0.9364670347299824,93.65
4000,Garry's Mod,Valve,891238,0.9663412017889722,96.63
252490,Rust,Facepunch Studios,844667,0.8672210468740936,86.72
550,Left 4 Dead 2,Valve,660664,0.9745286560187932,97.45


In [0]:
# Prepare the most-reviewed games for visualization
# ---------------------------------------------------------------------------

most_reviewed_chart_df = (
    most_reviewed_games_df
    .select(
        "name",
        "total_reviews"
    )
    .orderBy(F.desc("total_reviews"))
)

display(most_reviewed_chart_df)

name,total_reviews
Counter-Strike: Global Offensive,6730438
PUBG: BATTLEGROUNDS,2093876
Dota 2,1852811
Grand Theft Auto V,1442644
Tom Clancy's Rainbow Six Siege,1086157
Terraria,1037091
Team Fortress 2,903830
Garry's Mod,891238
Rust,844667
Left 4 Dead 2,660664


Databricks visualization. Run in Databricks to view.

### 7.3 Highest-Rated Games with Substantial Review Volume

In [0]:
# Rank games with substantial review evidence by player reception
# ---------------------------------------------------------------------------

MIN_REVIEWS = 750

top_rated_games_df = (
    steam_games_df
    .filter(F.col("total_reviews") >= MIN_REVIEWS)
    .select(
        "appid",
        "name",
        "publisher",
        "total_reviews",
        "positive_review_ratio"
    )
    .withColumn(
        "positive_review_pct",
        F.round(F.col("positive_review_ratio") * 100, 2)
    )
    .orderBy(
        F.desc("positive_review_ratio"),
        F.desc("total_reviews")
    )
    .limit(15)
)

display(top_rated_games_df)

appid,name,publisher,total_reviews,positive_review_ratio,positive_review_pct
858940,Flowers -Le volume sur ete-,JAST USA,938,0.9989339019189766,99.89
1530140,Aventura Copilului Albastru și Urât,Codrin Bradea,2217,0.9936851601262968,99.37
1677770,The Case of the Golden Idol,Playstack,915,0.9934426229508196,99.34
431730,Aseprite,Igara Studio,11903,0.9932790052927833,99.33
1260520,Patrick's Parabox,Patrick Traynor,1500,0.9926666666666667,99.27
1055540,A Short Hike,adamgryu,11732,0.9925843845891579,99.26
1665190,Monster Prom 3: Monster Roadtrip,Beautiful Glitch,1060,0.9924528301886792,99.25
1684930,CULTIC,3D Realms,2037,0.9921453117329406,99.21
1144400,Senren＊Banka,"HIKARI FIELD, NekoNyan Ltd.",10677,0.9921326215228997,99.21
1901370,Ib,PLAYISM,1829,0.9917987971569163,99.18


### 7.2–7.3 Key Findings — Popularity and Player Reception

Review volume and positive-review ratio reveal two different dimensions of videogame performance on Steam.

#### Games with the Highest Review Volume

**Counter-Strike: Global Offensive** is the most reviewed game in the dataset by a substantial margin, with more than **6.7 million recorded reviews**. It is followed by **PUBG: BATTLEGROUNDS** with approximately 2.09 million and **Dota 2** with approximately 1.85 million reviews.

Several other major titles exceed one million reviews, including **Grand Theft Auto V**, **Tom Clancy's Rainbow Six Siege**, and **Terraria**.

The presence of **Tom Clancy's Rainbow Six Siege** is particularly relevant from Ubisoft's perspective. With approximately **1.09 million reviews** and a positive-review ratio of **86.81%**, it ranks among the most extensively reviewed games in the complete Steam dataset. This demonstrates that Ubisoft already has at least one title with exceptionally high player engagement on the platform.

However, review volume and reception should not be treated as equivalent measures. For example, PUBG: BATTLEGROUNDS has more than two million reviews but a positive-review ratio of **56.61%**, whereas Terraria combines more than one million reviews with approximately **97.84% positive reception**. High player attention therefore does not necessarily imply equally strong player satisfaction.

#### Highest-Rated Games with Substantial Review Evidence

After restricting the analysis to games with at least **750 reviews**, the highest positive-review ratios remain extremely high. The leading titles in this subset receive approximately **99% positive reviews**, demonstrating that exceptionally strong player reception is possible even among games with substantial review evidence.

The highest-rated titles are not necessarily the games with the largest overall review volumes. Some highly rated games have only around one or two thousand reviews, while others, such as **Aseprite** and **A Short Hike**, combine positive-review ratios above 99% with more than 10,000 recorded reviews.

This distinction reinforces the need to analyze **popularity and reception separately**. Review volume provides a proxy for the scale of player attention, whereas positive-review ratio provides an indicator of how favorably players who submitted reviews evaluated the game.

Neither metric should be interpreted as a direct measure of revenue, profitability, or exact sales.

### 7.4 Ownership as a Popularity Proxy

The source dataset provides estimated ownership ranges rather than exact ownership counts. During data preparation, these ranges were transformed into lower and upper bounds and an approximate midpoint.

The ownership midpoint can provide an additional indicator of a game's potential commercial reach. However, it remains an approximation derived from a range and must not be interpreted as an exact number of owners, units sold, or generated revenue.

This analysis therefore uses ownership estimates only as a complementary popularity proxy alongside review volume and concurrent-user activity.

In [0]:
# Summarize the distribution of approximate ownership
# ---------------------------------------------------------------------------

ownership_summary_df = (
    steam_games_df
    .agg(
        F.expr(
            "percentile_approx(owner_midpoint, 0.5)"
        ).alias("median_owner_midpoint"),
        F.expr(
            "percentile_approx(owner_midpoint, 0.75)"
        ).alias("q3_owner_midpoint"),
        F.expr(
            "percentile_approx(owner_midpoint, 0.90)"
        ).alias("p90_owner_midpoint"),
        F.expr(
            "percentile_approx(owner_midpoint, 0.99)"
        ).alias("p99_owner_midpoint"),
        F.max("owner_midpoint").alias("max_owner_midpoint")
    )
)

display(ownership_summary_df)

median_owner_midpoint,q3_owner_midpoint,p90_owner_midpoint,p99_owner_midpoint,max_owner_midpoint
10000.0,35000.0,150000.0,1500000.0,3.5E8


In [0]:
# Identify games with the largest approximate ownership ranges
# ---------------------------------------------------------------------------

top_owned_games_df = (
    steam_games_df
    .select(
        "appid",
        "name",
        "publisher",
        "owner_lower_bound",
        "owner_upper_bound",
        "owner_midpoint",
        "total_reviews",
        "concurrent_users"
    )
    .orderBy(
        F.desc("owner_midpoint"),
        F.desc("total_reviews")
    )
    .limit(15)
)

display(top_owned_games_df)

appid,name,publisher,owner_lower_bound,owner_upper_bound,owner_midpoint,total_reviews,concurrent_users
570,Dota 2,Valve,200000000,500000000,3.5E8,1852811,852995
730,Counter-Strike: Global Offensive,Valve,50000000,100000000,7.5E7,6730438,874053
578080,PUBG: BATTLEGROUNDS,"KRAFTON, Inc.",50000000,100000000,7.5E7,2093876,339287
440,Team Fortress 2,Valve,50000000,100000000,7.5E7,903830,108900
1063730,New World,Amazon Games,50000000,100000000,7.5E7,238165,127379
271590,Grand Theft Auto V,Rockstar Games,20000000,50000000,3.5E7,1442644,140671
359550,Tom Clancy's Rainbow Six Siege,Ubisoft,20000000,50000000,3.5E7,1086157,31239
105600,Terraria,Re-Logic,20000000,50000000,3.5E7,1037091,58984
4000,Garry's Mod,Valve,20000000,50000000,3.5E7,891238,39043
252490,Rust,Facepunch Studios,20000000,50000000,3.5E7,844667,121146


### 7.4 Key Findings — Ownership and Commercial Reach

Estimated ownership is also highly concentrated across the Steam videogame catalogue.

The median ownership midpoint is approximately **10,000**, increasing to **35,000 at the 75th percentile**, **150,000 at the 90th percentile**, and **1.5 million at the 99th percentile**. This indicates that only a relatively small proportion of games reach ownership estimates in the millions.

At the upper end of the distribution, **Dota 2** is associated with the largest ownership range in the dataset, between **200 million and 500 million owners**, corresponding to an approximate midpoint of 350 million. Several other major titles, including Counter-Strike: Global Offensive, PUBG: BATTLEGROUNDS, Team Fortress 2, and New World, fall within the 50–100 million ownership range.

Many of the games with the highest ownership estimates also appear among the most reviewed titles. Counter-Strike: Global Offensive, PUBG: BATTLEGROUNDS, Grand Theft Auto V, Terraria, Rust, and other highly reviewed games are represented in the upper ownership ranges. This provides qualitative consistency between two independent popularity indicators in the dataset.

From Ubisoft's perspective, **Tom Clancy's Rainbow Six Siege** is particularly notable. It falls within the estimated **20–50 million ownership range**, while also recording more than **1.08 million reviews**. Together, these indicators suggest substantial reach and player engagement on Steam.

However, ownership values are provided as broad intervals rather than exact counts. Games within the same interval therefore receive identical derived midpoints, and the midpoint should not be interpreted as a precise ownership estimate. For this reason, ownership is used as a supporting indicator of relative commercial reach rather than as an exact sales metric.

### 7.5 Concurrent Users as an Engagement Indicator

Concurrent-user activity provides an additional perspective on player engagement by capturing the number of simultaneously active users represented in the dataset.

Unlike cumulative review volume and ownership estimates, concurrent users reflect activity at a particular observation point. The metric therefore favors games that were actively played when the dataset was collected and should not be interpreted as a historical measure of lifetime popularity.

In [0]:
# Identify games with the highest concurrent-user activity
# ---------------------------------------------------------------------------

top_concurrent_games_df = (
    steam_games_df
    .select(
        "appid",
        "name",
        "publisher",
        "concurrent_users",
        "total_reviews",
        "owner_midpoint"
    )
    .orderBy(F.desc("concurrent_users"))
    .limit(15)
)

display(top_concurrent_games_df)

appid,name,publisher,concurrent_users,total_reviews,owner_midpoint
730,Counter-Strike: Global Offensive,Valve,874053,6730438,7.5E7
570,Dota 2,Valve,852995,1852811,3.5E8
578080,PUBG: BATTLEGROUNDS,"KRAFTON, Inc.",339287,2093876,7.5E7
1172470,Apex Legends,Electronic Arts,314468,527675,3.5E7
1599340,Lost Ark,Amazon Games,273088,174357,3.5E7
1938090,Call of Duty: Modern Warfare II,Activision,206441,62830,3500000.0
271590,Grand Theft Auto V,Rockstar Games,140671,1442644,3.5E7
1063730,New World,Amazon Games,127379,238165,7.5E7
252490,Rust,Facepunch Studios,121146,844667,3.5E7
1203220,NARAKA: BLADEPOINT,NetEase Games Global,116729,131469,7500000.0


In [0]:
# Summarize concurrent-user activity across the catalogue
# ---------------------------------------------------------------------------

concurrent_users_summary_df = (
    steam_games_df
    .agg(
        F.sum(
            F.when(F.col("concurrent_users") == 0, 1).otherwise(0)
        ).alias("games_with_zero_ccu"),
        F.expr(
            "percentile_approx(concurrent_users, 0.5)"
        ).alias("median_ccu"),
        F.expr(
            "percentile_approx(concurrent_users, 0.90)"
        ).alias("p90_ccu"),
        F.expr(
            "percentile_approx(concurrent_users, 0.99)"
        ).alias("p99_ccu"),
        F.max("concurrent_users").alias("max_ccu")
    )
)

display(concurrent_users_summary_df)

games_with_zero_ccu,median_ccu,p90_ccu,p99_ccu,max_ccu
39597,0,8,802,874053


### 7.5 Key Findings — Concurrent Player Activity

Concurrent-user activity is extremely concentrated among a relatively small number of Steam games.

Of the **55,690 videogames** in the analytical dataset, **39,597 have zero recorded concurrent users** at the observation point. The median concurrent-user count is therefore **0**, while even the 90th percentile reaches only **8 concurrent users**. At the 99th percentile, however, activity rises to **802 users**, demonstrating the strongly skewed nature of player activity across the catalogue.

At the upper extreme, **Counter-Strike: Global Offensive** records the highest concurrent-user activity with **874,053 users**, closely followed by **Dota 2 with 852,995**. PUBG: BATTLEGROUNDS, Apex Legends, Lost Ark, and Call of Duty: Modern Warfare II also show particularly high activity.

Several games appearing in this ranking also rank highly by review volume or estimated ownership, including Counter-Strike: Global Offensive, Dota 2, PUBG: BATTLEGROUNDS, Grand Theft Auto V, Rust, and Team Fortress 2. This provides additional evidence that player attention on Steam is concentrated among a comparatively small group of highly visible titles.

However, concurrent-user counts represent activity at a particular observation point rather than lifetime performance. A zero value does not necessarily imply that a game has never attracted players, and differences between games may also reflect release timing, time of day, regional activity, or lifecycle stage.

Concurrent users are therefore interpreted as a **snapshot engagement indicator**, complementary to cumulative review volume and approximate ownership rather than as a standalone measure of commercial success.

## 8. Market Analysis — Key Business Insights

The market-level analysis highlights several characteristics of the Steam ecosystem that are relevant when considering the positioning of a future Ubisoft release.

### Increasing Competition for Visibility

The number of videogames released annually increased substantially from the mid-2010s onward, reaching a dataset peak of **8,805 releases in 2021**. This indicates a highly active marketplace in which new releases compete with thousands of other titles for player attention.

The lower count observed in 2022 should not be interpreted as evidence of market contraction because the dataset covers only part of that calendar year.

### A Highly Fragmented Publisher Ecosystem

Steam contains a very large and diverse publisher population. The dataset includes **29,833 distinct publisher entries** among games with publisher information, while even the most represented publisher accounts for fewer than 1% of those games.

Ubisoft nevertheless has a substantial catalogue presence, ranking among the most represented publisher entries with **127 games**.

This suggests that competition on Steam extends beyond a small group of major publishers to a broad ecosystem of independent and specialized publishers.

### Paid Games Dominate, but Free-to-Play Is Established

Paid titles represent **86.03%** of the catalogue, compared with **13.97%** classified as initially free.

Among paid games, the median initial price is **5.99**, while the average is higher at **9.27**, reflecting a right-skewed price distribution. The middle 50% of paid titles are priced between **2.99 and 11.99**.

These catalogue-wide figures provide market context rather than an optimal pricing recommendation for Ubisoft, since appropriate pricing depends on product characteristics and positioning.

### Discounts Are Selective but Substantial

Only **4.52%** of games are discounted in the dataset snapshot. Among discounted titles, however, the median discount is **60%** and the average is **57.59%**.

This indicates that active promotions can involve substantial price reductions, although the snapshot nature of the dataset prevents conclusions about historical discount frequency, duration, or optimal promotional timing.

### Broad Localization Is Not Universal

Although games support an average of **3.63 languages**, the median is only **1 language**. English is nearly universal among games with language information, while substantially fewer games support additional languages.

For an international publisher such as Ubisoft, broad localization can therefore represent an important accessibility and positioning dimension, particularly when targeting multiple geographic markets.

### Player Attention Is Highly Concentrated

Review volume, ownership estimates, and concurrent-user activity all indicate a strongly concentrated market.

The median game has only **26 reviews**, compared with an average of more than 1,700, while the 99th percentile exceeds 25,000 reviews. Similarly, the median ownership midpoint is approximately **10,000**, compared with **1.5 million at the 99th percentile**.

Concurrent activity is even more concentrated: **39,597 games have zero recorded concurrent users**, while a small number of leading titles reach hundreds of thousands.

Together, these indicators suggest that the distribution of player attention on Steam is highly unequal: a comparatively small group of games attracts a disproportionate amount of engagement.

### Ubisoft Already Demonstrates Strong Steam Reach

**Tom Clancy's Rainbow Six Siege** provides a notable Ubisoft benchmark within the dataset. The game records approximately **1.09 million reviews**, an **86.81% positive-review ratio**, and an estimated ownership range of **20–50 million**.

Its presence among the most reviewed games demonstrates that Ubisoft is capable of achieving substantial visibility and player engagement on Steam.

However, the broader market results also show that high visibility is exceptional rather than typical. A future release should therefore be evaluated not only against overall catalogue averages but also in relation to its genre, platform availability, and product positioning.

## 9. Next Steps

The market analysis established the overall competitive context of the Steam videogame ecosystem.

The next notebook moves from market-level characteristics to **product-level positioning**, focusing on:

- genre representation across the catalogue;
- genre-level player reception and popularity;
- publisher specialization where analytically useful;
- Windows, macOS, and Linux availability;
- cross-platform patterns;
- relationships between genre, platform support, pricing, reviews, and popularity indicators.

Genre analysis will require transforming the multi-valued genre arrays using Spark's `explode()` operation. Unique-game counts will be preserved carefully after this transformation to avoid double-counting games associated with multiple genres.

The resulting analysis will complement the market-level findings and support the final business recommendations for Ubisoft.